In [ ]:
    def save_weights(self, time=None):
        """
        Saves the synaptic weights of the STDP synapses in a file.
        """

        if self.simulation_params["recording_params"]["save_weights"]:
            conn = nest.GetConnections(synapse_model="stdp_synapse_rec")
            
            sources = conn.source
            targets = conn.target
            weights = conn.weight
            
            data = np.column_stack((sources, targets, weights))
            
            filepath = self.simulation_params['data_path'] + "weights" + str(time) + ".dat" if time is not None else self.simulation_params['data_path'] + "final_weights.dat"
            
            np.savetxt(filepath, data, fmt=['%d', '%d', '%.6f'], header="Source Target Weight")
            
            print(f"weights saved to: {filepath}")

In [ ]:
    def connect_populations(self):
        """
        Creation of the connections between neuron populations.

        """

        #print option to be implemented
        more_print = False

        print("Connecting the neuron populations...", end = ' ')
        for i in range(self.p):
            if more_print:
                print("\nTarget: selective population ", i+1)
            # indegrees from the exc selective pops
            for j in range(self.p):
                if more_print:
                    print("\tSource: selective population ", j+1)
                if i==j:
                    # Creation of synapses between same selective populations
                    con_dict = {'rule': 'fixed_indegree', 
                            'indegree': int(self.f*self.c*self.network_params["N_exc"]),
                            'allow_autapses': self.network_params["syn_params"]["autapses"], 'allow_multapses': self.network_params["syn_params"]["multapses"]}
                    
                    syn_dict = {"synapse_model": "stdp_synapse_rec",
                                "tau_plus": self.network_params["stdp_params"]["tau_plus"],
                                "lambda": self.network_params["stdp_params"]["lambda"],
                                "alpha": self.network_params["stdp_params"]["alpha"],
                                "mu_plus": self.network_params["stdp_params"]["mu_plus"],
                                "mu_minus": self.network_params["stdp_params"]["mu_minus"],
                                "Wmax": self.network_params["stdp_params"]["Wmax"],
                                "delay": nest.random.uniform(min=self.network_params["syn_params"]["delay"][0], max=self.network_params["syn_params"]["delay"][1]),
                                "weight": get_weight(self.network_params["syn_params"]["J_p"], self.network_params["neur_params"]["tau"][0])}
                    nest.Connect(self.exc_populations[j], self.exc_populations[i], con_dict, syn_dict)
                else:
                    # Creation of synapses between different selective populations
                    con_dict = {'rule': 'fixed_indegree', 
                            'indegree': int(self.f*self.c*self.network_params["N_exc"]),
                            'allow_autapses': self.network_params["syn_params"]["autapses"], 'allow_multapses': self.network_params["syn_params"]["multapses"]}
                
                    syn_dict = {"synapse_model": "stdp_synapse_rec",
                                "tau_plus": self.network_params["stdp_params"]["tau_plus"],
                                "lambda": self.network_params["stdp_params"]["lambda"],
                                "alpha": self.network_params["stdp_params"]["alpha"],
                                "mu_plus": self.network_params["stdp_params"]["mu_plus"],
                                "mu_minus": self.network_params["stdp_params"]["mu_minus"],
                                "Wmax": self.network_params["stdp_params"]["Wmax"],
                                "delay": nest.random.uniform(min=self.network_params["syn_params"]["delay"][0], max=self.network_params["syn_params"]["delay"][1]),
                                "weight": get_weight(self.network_params["syn_params"]["J_b"], self.network_params["neur_params"]["tau"][0])}
                    nest.Connect(self.exc_populations[j], self.exc_populations[i], con_dict, syn_dict)
            
            # indegrees from the other exc neurons
            if more_print:
                print("\tSource: non-selective exc pop")
            #Creation of synapses from the other exc neurons
            con_dict = {'rule': 'fixed_indegree', 'indegree': int((1.0-self.network_params["syn_params"]["gamma_0"])*self.c*(1.0-self.f*self.p)*self.network_params["N_exc"]),
                        'allow_autapses': self.network_params["syn_params"]["autapses"], 'allow_multapses': self.network_params["syn_params"]["multapses"]}
            
            syn_dict = {"synapse_model": "stdp_synapse_rec",
                                "tau_plus": self.network_params["stdp_params"]["tau_plus"],
                                "lambda": self.network_params["stdp_params"]["lambda"],
                                "alpha": self.network_params["stdp_params"]["alpha"],
                                "mu_plus": self.network_params["stdp_params"]["mu_plus"],
                                "mu_minus": self.network_params["stdp_params"]["mu_minus"],
                                "Wmax": self.network_params["stdp_params"]["Wmax"],
                                "delay": nest.random.uniform(min=self.network_params["syn_params"]["delay"][0], max=self.network_params["syn_params"]["delay"][1]),
                                "weight": get_weight(self.network_params["syn_params"]["J_b"], self.network_params["neur_params"]["tau"][0])}
            nest.Connect(self.exc_populations[-1], self.exc_populations[i], con_dict, syn_dict)

            if(self.network_params["syn_params"]["gamma_0"] > 0.0):
                con_dict['indegree'] = int(self.network_params["syn_params"]["gamma_0"]*self.c*(1.0-self.f*self.p)*self.network_params["N_exc"])

                syn_dict["weight"] = get_weight(self.network_params["syn_params"]["J_p"], self.network_params["neur_params"]["tau"][0])
                nest.Connect(self.exc_populations[-1], self.exc_populations[i], con_dict, syn_dict)

            # indegrees from the inh pop
            if more_print:
                print("\tSource: inh pop")
            con_dict = {'rule': 'fixed_indegree', 'indegree': int(self.c*self.network_params["N_inh"]),
                        'allow_autapses': self.network_params["syn_params"]["autapses"], 'allow_multapses': self.network_params["syn_params"]["multapses"]}
            syn_dict = {"synapse_model": "static_synapse",
                        "weight": get_weight(-self.network_params["syn_params"]["J_EI"], self.network_params["neur_params"]["tau"][1]),
                        "delay": nest.random.uniform(min=self.network_params["syn_params"]["delay"][0], max=self.network_params["syn_params"]["delay"][1])}
            nest.Connect(self.inh_population, self.exc_populations[i], con_dict, syn_dict)

        # connections for the inhibitory population
        if more_print:
            print("\nTarget: inhibitory population")
        # indegrees from the populations
        for i in range(self.p):
            if more_print:
                print("\tSource: selective population ", i+1)
            con_dict = {'rule': 'fixed_indegree', 'indegree': int(self.f*self.c*self.network_params["N_exc"]),
                        'allow_autapses': self.network_params["syn_params"]["autapses"], 'allow_multapses': self.network_params["syn_params"]["multapses"]}
            syn_dict = {"synapse_model": "static_synapse",
                        "weight": get_weight(self.network_params["syn_params"]["J_IE"], self.network_params["neur_params"]["tau"][1]),
                        "delay": nest.random.uniform(min=self.network_params["syn_params"]["delay"][0], max=self.network_params["syn_params"]["delay"][1])}    
            nest.Connect(self.exc_populations[i], self.inh_population, con_dict, syn_dict)

        # indegrees from the other exc neurons
        if more_print:
            print("\tSource: non-selective exc pop")
        con_dict = {'rule': 'fixed_indegree', 'indegree': int(self.c*(1.0-self.f*self.p)*self.network_params["N_exc"]),
                    'allow_autapses': self.network_params["syn_params"]["autapses"], 'allow_multapses': self.network_params["syn_params"]["multapses"]}
        syn_dict = {"synapse_model": "static_synapse",
                    "weight": get_weight(self.network_params["syn_params"]["J_IE"], self.network_params["neur_params"]["tau"][0]),
                    "delay": nest.random.uniform(min=self.network_params["syn_params"]["delay"][0], max=self.network_params["syn_params"]["delay"][1])}  
        nest.Connect(self.exc_populations[-1], self.inh_population, con_dict, syn_dict)

        # indegrees from inh pop itself
        if more_print:
            print("\tSource: inh pop")
        con_dict = {'rule': 'fixed_indegree', 'indegree': int(self.c*self.network_params["N_inh"]),
                    'allow_autapses': self.network_params["syn_params"]["autapses"], 'allow_multapses': self.network_params["syn_params"]["multapses"]}
        syn_dict = {"synapse_model": "static_synapse",
                    "weight": get_weight(-self.network_params["syn_params"]["J_II"], self.network_params["neur_params"]["tau"][1]),
                    "delay": nest.random.uniform(min=self.network_params["syn_params"]["delay"][0], max=self.network_params["syn_params"]["delay"][1])} 
        nest.Connect(self.inh_population, self.inh_population, con_dict, syn_dict)

        # connection for the non-specific exc population
        if more_print:
            print("\nTarget: non-specific excitatory population")
        # indegrees from the selective populations
        for i in range(self.p):
            if more_print:
                print("\tSource: selective population ", i+1)
            con_dict = {'rule': 'fixed_indegree', 'indegree': int(self.f*self.c*self.network_params["N_exc"]),
                        'allow_autapses': self.network_params["syn_params"]["autapses"], 'allow_multapses': self.network_params["syn_params"]["multapses"]}
            
            syn_dict = {"synapse_model": "stdp_synapse_rec",
                                "tau_plus": self.network_params["stdp_params"]["tau_plus"],
                                "lambda": self.network_params["stdp_params"]["lambda"],
                                "alpha": self.network_params["stdp_params"]["alpha"],
                                "mu_plus": self.network_params["stdp_params"]["mu_plus"],
                                "mu_minus": self.network_params["stdp_params"]["mu_minus"],
                                "Wmax": self.network_params["stdp_params"]["Wmax"],
                                "delay": nest.random.uniform(min=self.network_params["syn_params"]["delay"][0], max=self.network_params["syn_params"]["delay"][1]),
                                "weight": get_weight(self.network_params["syn_params"]["J_b"], self.network_params["neur_params"]["tau"][0])}
            nest.Connect(self.exc_populations[i], self.exc_populations[-1], con_dict, syn_dict)

        # indegrees from the rest of the exc pop
        if more_print:
            print("\tSource: non-selective exc pop")
        con_dict = {'rule': 'fixed_indegree', 'indegree': int((1.0-self.network_params["syn_params"]["gamma_0"])*self.c*(1.0-self.f*self.p)*self.network_params["N_exc"]),
                    'allow_autapses': self.network_params["syn_params"]["autapses"], 'allow_multapses': self.network_params["syn_params"]["multapses"]}
        
        syn_dict = {"synapse_model": "stdp_synapse_rec",
                                "tau_plus": self.network_params["stdp_params"]["tau_plus"],
                                "lambda": self.network_params["stdp_params"]["lambda"],
                                "alpha": self.network_params["stdp_params"]["alpha"],
                                "mu_plus": self.network_params["stdp_params"]["mu_plus"],
                                "mu_minus": self.network_params["stdp_params"]["mu_minus"],
                                "Wmax": self.network_params["stdp_params"]["Wmax"],
                                "delay": nest.random.uniform(min=self.network_params["syn_params"]["delay"][0], max=self.network_params["syn_params"]["delay"][1]),
                                "weight": get_weight(self.network_params["syn_params"]["J_b"], self.network_params["neur_params"]["tau"][0])}
        nest.Connect(self.exc_populations[-1], self.exc_populations[-1], con_dict, syn_dict)

        if(self.network_params["syn_params"]["gamma_0"] > 0.0):
            con_dict = {'rule': 'fixed_indegree', 'indegree': int(self.network_params["syn_params"]["gamma_0"]*self.c*(1.0-self.f*self.p)*self.network_params["N_exc"]),
                        'allow_autapses': self.network_params["syn_params"]["autapses"], 'allow_multapses': self.network_params["syn_params"]["multapses"]}
            
            syn_dict = {"synapse_model": "stdp_synapse_rec",
                                    "tau_plus": self.network_params["stdp_params"]["tau_plus"],
                                    "lambda": self.network_params["stdp_params"]["lambda"],
                                    "alpha": self.network_params["stdp_params"]["alpha"],
                                    "mu_plus": self.network_params["stdp_params"]["mu_plus"],
                                    "mu_minus": self.network_params["stdp_params"]["mu_minus"],
                                    "Wmax": self.network_params["stdp_params"]["Wmax"],
                                    "delay": nest.random.uniform(min=self.network_params["syn_params"]["delay"][0], max=self.network_params["syn_params"]["delay"][1]),
                                    "weight": get_weight(self.network_params["syn_params"]["J_p"], self.network_params["neur_params"]["tau"][0])}
            nest.Connect(self.exc_populations[-1], self.exc_populations[-1], con_dict, syn_dict)

        # indegrees from the inh pop
        if more_print:
            print("\tSource: inh pop")
        con_dict = {'rule': 'fixed_indegree', 'indegree': int(self.c*self.network_params["N_inh"]),
                    'allow_autapses': self.network_params["syn_params"]["autapses"], 'allow_multapses': self.network_params["syn_params"]["multapses"]}
        syn_dict = {"synapse_model": "static_synapse",
                    "weight": get_weight(-self.network_params["syn_params"]["J_EI"], self.network_params["neur_params"]["tau"][1]),
                    "delay": nest.random.uniform(min=self.network_params["syn_params"]["delay"][0], max=self.network_params["syn_params"]["delay"][1])}
        nest.Connect(self.inh_population, self.exc_populations[-1], con_dict, syn_dict)

        print("Done")

In [ ]:
    def save_weights(self, time:float=0.0):
        """
        Saves synaptic weights in a single file named 'weights_timeX.dat' where X is the time.

        Args:
            time (float): time at which the weights are saved (in ms). Defaults to 0.0.
        """

        if time is not None:
            filepath = self.simulation_params['data_path'] + "weights_" + str(time) + ".dat"
        else:
            filepath = self.simulation_params['data_path'] + "final_weights.dat"

        with open(filepath, 'w') as f:
            
            for i in range(self.p):
                for j in range(self.p):
                    # weights from selective population j to selective population i
                    header = "weights_selective_pop{}_to_selective_pop{}\n".format(i, j)
                    
                    f.write(header)

                    w = nest.GetStatus(nest.GetConnections(self.exc_populations[i], self.exc_populations[j], synapse_model="stdp_synapse_rec"), "weight")

                    weights_string = " ".join(map(str, w)) + "\n"
                    f.write(weights_string)
                
                # weights from non-selective population to selective population i
                header = "weights_nonselective_pop_to_selective_pop{}\n".format(i)

                f.write(header)

                w = nest.GetStatus(nest.GetConnections(self.exc_populations[-1], self.exc_populations[i], synapse_model="stdp_synapse_rec"), "weight")

                weights_string = " ".join(map(str, w)) + "\n"
                f.write(weights_string)

                # weights from selective population i to non-selective population
                header = "weights_selective_pop{}_to_nonselective_pop\n".format(i)
                f.write(header)

                w = nest.GetStatus(nest.GetConnections(self.exc_populations[i], self.exc_populations[-1], synapse_model="stdp_synapse_rec"), "weight")

                weights_string = " ".join(map(str, w)) + "\n"
                f.write(weights_string)

            # weights from non-selective population to non-selective population
            header = "weights_nonselective_pop_to_nonselective_pop\n"

            f.write(header)

            w = nest.GetStatus(nest.GetConnections(self.exc_populations[-1], self.exc_populations[-1], synapse_model="stdp_synapse_rec"), "weight")
                
            weights_string = " ".join(map(str, w)) + "\n"
            f.write(weights_string)


In [ ]:
    def save_weights(self, time:float=0.0):
        """
        Saves synaptic weights in a single file named 'weights_timeX.dat' where X is the time.
        """
        import numpy as np

        if time is not None:
            filepath = self.simulation_params['data_path'] + "weights_" + str(time) + ".dat"
        else:
            filepath = self.simulation_params['data_path'] + "final_weights.dat"

        # How many neurons to analyze at once (chunk size)
        chunk_size = 100 

        # Inner helper function to process connections in chunks and prevent RAM saturation
        def write_connections_chunked(file_handle, sources, targets):
            # Iterate over source neurons in chunks (e.g., 0 to 100, 100 to 200...)
            for k in range(0, len(sources), chunk_size):
                src_chunk = sources[k : k + chunk_size]
                
                # Request connections from NEST only for this specific chunk of neurons
                conns = nest.GetConnections(src_chunk, targets, synapse_model="stdp_synapse_rec")
                
                if len(conns) > 0:
                    w = np.array(conns.get("weight"))
                    
                    # .tofile() writes the array quickly to the open file, 
                    # using space as a separator. It avoids loading huge strings in memory.
                    w.tofile(file_handle, sep=" ", format="%.4f")
                    
                    # Add an empty space to separate this chunk from the next one
                    file_handle.write(" ")
            
            # Once all chunks for the current population pair are done, add a newline
            file_handle.write("\n")

        # FILE OPENING AND ITERATION OVER POPULATION COMBINATIONS
        with open(filepath, 'w') as f:
            for i in range(self.p):
                for j in range(self.p):
                    # From Selective population J to Selective population I
                    f.write("weights_selective_pop{}_to_selective_pop{}\n".format(i, j))
                    write_connections_chunked(f, self.exc_populations[i], self.exc_populations[j])

                # From Non-selective population to Selective population I
                f.write("weights_nonselective_pop_to_selective_pop{}\n".format(i))
                write_connections_chunked(f, self.exc_populations[-1], self.exc_populations[i])

                # From Selective population I to Non-selective population
                f.write("weights_selective_pop{}_to_nonselective_pop\n".format(i))
                write_connections_chunked(f, self.exc_populations[i], self.exc_populations[-1])

            # From Non-selective population to Non-selective population
            f.write("weights_nonselective_pop_to_nonselective_pop\n")
            write_connections_chunked(f, self.exc_populations[-1], self.exc_populations[-1])


In [ ]:
def stop_stdp_chunked(self, sources, targets, chunk_size:int=10):
        """
        Get the connections between source and target populations in chunks to avoid memory issues.

        Args:
            sources (list): list of source neurons.
            targets (list): list of target neurons.
            chunk_size (int): size of the chunks to get the connections.
        """

        for k in range(0, len(sources), chunk_size):
            src_chunk = sources[k : k + chunk_size]

            conns = nest.GetConnections(src_chunk, targets, synapse_model="stdp_synapse_rec")
            nest.SetStatus(conns, {"lambda": 0.0})

    def stop_stdp(self):
        """
        Stop STDP plasticity by setting the learning rate to 0.

        """
        print("Stopping the STDP...")

        for i in range(self.p):
            for j in range(self.p):
                # From Selective population J to Selective population I
                self.stop_stdp_chunked(self.exc_populations[i], self.exc_populations[j])

            # From Non-selective population to Selective population I
            self.stop_stdp_chunked(self.exc_populations[-1], self.exc_populations[i])

            # From Selective population I to Non-selective population
            self.stop_stdp_chunked(self.exc_populations[i], self.exc_populations[-1])

        # From Non-selective population to Non-selective population
        self.stop_stdp_chunked(self.exc_populations[-1], self.exc_populations[-1])


        print("STDP stopped.")